In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [7]:
if is_main:
    seed = 43
    environment_string = "boxing"
    gold_timesteps = 30_000_000
    training_timesteps = 100_000 
    num_concepts_selected = 50
    out_folder = "basic"
    method = "greedy" 


In [8]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed,processed_concepts=processed_concepts)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,concept_idx=list(range(len(concept_list))),processed_concepts=processed_concepts)
    two_stage_env.reset().shape

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [9]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

In [14]:
concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,groundtruth_model,concept_list,list(range(len(concept_list))),environment_string,epochs=25,max_episode_length=10_000)

KeyboardInterrupt: 

In [8]:
model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
if os.path.exists(model_name):
    q_estimates = pickle.load(open(model_name,"rb"))
else:
    q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    pickle.dump(q_estimates,open(model_name,"wb"))


Starting stable training for sparse rewards...
Total of 100000 steps
Step 0/100000, Loss mean: 0.0000
Episode 25, Avg Reward: 131.96, Loss (mean/std/max): 0.06/0.04/0.22, Epsilon: 0.092
Step 5000/100000, Loss mean: 0.0258
Episode 50, Avg Reward: 191.60, Loss (mean/std/max): 0.02/0.00/0.03, Epsilon: 0.081
Step 10000/100000, Loss mean: 0.0145
Updated target network at step 500, Avg recent loss: 0.0144
Episode 75, Avg Reward: 155.92, Loss (mean/std/max): 0.19/0.29/0.99, Epsilon: 0.074
Step 15000/100000, Loss mean: 0.0147
Episode 100, Avg Reward: 157.48, Loss (mean/std/max): 0.01/0.01/0.04, Epsilon: 0.067
Episode 125, Avg Reward: 142.76, Loss (mean/std/max): 0.01/0.01/0.06, Epsilon: 0.061
Step 20000/100000, Loss mean: 0.0132
Updated target network at step 1000, Avg recent loss: 0.0134
Step 25000/100000, Loss mean: 0.0317
Step 30000/100000, Loss mean: 0.0211
Updated target network at step 1500, Avg recent loss: 0.0217
Episode 150, Avg Reward: 470.12, Loss (mean/std/max): 0.16/0.25/0.99, Eps

In [26]:
groundtruth_model.observation_space

Box(0, 255, (4, 160, 240), uint8)

In [22]:
concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,groundtruth_model,concept_list,list(range(len(concept_list))),environment_string,epochs=25,max_episode_length=10_000)

ValueError: Error: Unexpected observation shape (4, 4, 160, 240) for Box environment, please use (4, 84, 84) or (n_env, 4, 84, 84) for the observation shape.

In [18]:
two_stage_env.reset()

array([[1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0.],
       [1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0.],
       [1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0.],
       [1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0.],
       [1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0.],
       [1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0.]], dtype=float32)

In [6]:
if is_main:
    start = time.time()
    total_timesteps = 0
    while total_timesteps <= 25_000:
        two_stage_env.step([1 for i in range(8)])
        total_timesteps += 8
    print("Time {}".format(time.time()-start))

Time 11.702568292617798


In [7]:
train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=25_000,custom_name="{}_perfect_random".format(environment_string))        

wandb: Currently logged in as: naveenr (naveenr-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


approx_kl,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▄▄▄▄▄▄▄▄▄▄██████████████
clip_fraction,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃██████████
ema_norm_reward,▁
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃██████████████
episode_length_mean,▁
episode_reward_max,▁
episode_reward_mean,▁
episode_reward_min,▁
episodes_completed,▁
explained_variance,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
+1,...
